In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
sys.path.insert(0, project_root)

In [2]:
import logging
import os

from dotenv import load_dotenv
from opensearchpy import OpenSearch

from src.retrieval.embedder import Embedder
from src.retrieval.searcher import HybridSearcher


In [3]:
load_dotenv()

True

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [5]:
client = OpenSearch(
    hosts=[{
        "host": os.getenv("OPENSEARCH_HOST", "localhost"),
        "port": int(os.getenv("OPENSEARCH_PORT", 9200))
    }],
    http_compress=True
)

embedder = Embedder(ollama_url=os.getenv("OLLAMA_URL"))
searcher = HybridSearcher(client=client, embedder=embedder, top_k=5)

In [6]:
query = "Qual o tratamento para crise aguda de angioedema hereditário?"
results = searcher.search(query)

print(f"\nQuery: {query}")
print(f"{'='*60}")
for i, result in enumerate(results):
    print(f"\n[{i+1}] Score: {result.score:.4f}")
    print(f"     Source: {result.source}")
    print(f"     Tokens: {result.metadata['token_count']}")
    print(f"     Text: {result.text[:200]}")

2026-07-29 23:53:33,491 - src.retrieval.searcher - INFO - Searching for: 'Qual o tratamento para crise aguda de angioedema hereditário?'
2026-07-29 23:53:33,778 - opensearch - INFO - POST http://localhost:9200/angioedema/_search [status:200 request:0.020s]
2026-07-29 23:53:33,838 - opensearch - INFO - POST http://localhost:9200/angioedema/_search [status:200 request:0.059s]
2026-07-29 23:53:33,841 - src.retrieval.searcher - INFO - Semantic hits: 5 | Keyword hits: 5
2026-07-29 23:53:33,842 - src.retrieval.searcher - INFO - Returning 5 results after RRF fusion



Query: Qual o tratamento para crise aguda de angioedema hereditário?

[1] Score: 0.0167
     Source: ASBAI - O que é Angioedema.pdf
     Tokens: 510
     Text: .br/pacientes.php
73 | Doutor, eu tenho Angioedema Hereditário

Doutor,
existe um dia para o AEH?
im. O dia 16 de maio é o “Dia Mundial
de Conscientização do Angioedema
Hereditário”.
74 | Doutor, eu t

[2] Score: 0.0167
     Source: ANVISA - Icatibanto.pdf
     Tokens: 645
     Text: a análise
tenha refletido, de forma objetiva, tanto o cuidado atual oferecido aos pacientes, quando os
benefícios tangíveis de icatibanto, um medicamento eficaz e seguro para o tratamento de crises
de

[3] Score: 0.0164
     Source: ASBAI - O que é Angioedema.pdf
     Tokens: 597
     Text: promovendo atividades direcionadas para o Angioedema
Hereditário. Realiza projetos educativos de esclarecimento
sobre a doença, capacitando profissionais e Serviços de Re-
ferência no Brasil.
A página

[4] Score: 0.0164
     Source: ANVISA - Icatibanto.pdf
     

In [ ]:
from src.retrieval.reranker import Reranker

reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=5)

searcher_wide = HybridSearcher(client=client, embedder=embedder, top_k=20)

query = "Qual o tratamento para crise aguda de angioedema hereditário?"

results = searcher_wide.search(query)
reranked = reranker.rerank(query, results)

print(f"\nQuery: {query}")
print(f"{'='*60}")
for i, result in enumerate(reranked):
    print(f"\n[{i+1}] Score: {result.score:.4f}")
    print(f"     Source: {result.source}")
    print(f"     Tokens: {result.metadata['token_count']}")
    print(f"     Text: {result.text[:300]}")

In [ ]:
from src.ingestion.loader import load_pdf

doc_ebserh = load_pdf("docs/raw/EBSERH - Protocolo Assistencial.pdf")
print(doc_ebserh.content)

2026-07-29 23:53:48,950 - src.ingestion.loader - INFO - Loading PDF: EBSERH - Protocolo Assistencial.pdf
2026-07-29 23:53:48,981 - src.ingestion.loader - INFO - Extracted: 8/8 pages with text from EBSERH - Protocolo Assistencial.pdf


PROTOCOLO ASSISTENCIAL
Tema: manejo da crise do angioedema hereditário
Protocolo Nº  232
1ª Versão: fevereiro de 2020
Versão Nº 01
Atualização: NA
SUMÁRIO

Manejo da crise do angioedema hereditário
INTRODUÇÃO
O angioedema hereditário (AEH) é um angioedema recorrente causado por excesso de
bradicinina, e é manifestação de uma síndrome geneticamente determinada por herança
autossômica dominante que se caracteriza pela deficiência quantitativa e/ou funcional do
inibidor do complemento 1 (C1-INH) e acarreta crises de edema localizado, não inflamatório,
com acometimento de diversos órgãos.

Na maioria das ocorrências, essas crises envolvem extremidades, abdome, trato genito-
urinário, orofaringe e laringe. Seu tratamento deve ser feito de acordo com sua gravidade
(ver “abordagem das crises”). Crises graves, que envolvem ou não o trato respiratório,
requerem tratamento urgente por causa das altas taxas de morbidade e mortalidade.
OBJETIVOS
O objetivo deste protocolo é sistematizar a abordage

In [11]:
from importlib import reload
import src.ingestion.loader as loader_module
reload(loader_module)
from src.ingestion.loader import load_pdf

doc_ebserh = load_pdf("docs/raw/EBSERH - Protocolo Assistencial.pdf")
print(doc_ebserh.content)

2026-07-29 23:53:55,003 - src.ingestion.loader - INFO - Loading PDF: EBSERH - Protocolo Assistencial.pdf
2026-07-29 23:53:55,033 - src.ingestion.loader - INFO - Extracted: 8/8 pages with text from EBSERH - Protocolo Assistencial.pdf


PROTOCOLO ASSISTENCIAL
Tema: manejo da crise do angioedema hereditário
Protocolo Nº  232
1ª Versão: fevereiro de 2020
Versão Nº 01
Atualização: NA
SUMÁRIO

Manejo da crise do angioedema hereditário
INTRODUÇÃO
O angioedema hereditário (AEH) é um angioedema recorrente causado por excesso de
bradicinina, e é manifestação de uma síndrome geneticamente determinada por herança
autossômica dominante que se caracteriza pela deficiência quantitativa e/ou funcional do
inibidor do complemento 1 (C1-INH) e acarreta crises de edema localizado, não inflamatório,
com acometimento de diversos órgãos.

Na maioria das ocorrências, essas crises envolvem extremidades, abdome, trato genito-
urinário, orofaringe e laringe. Seu tratamento deve ser feito de acordo com sua gravidade
(ver “abordagem das crises”). Crises graves, que envolvem ou não o trato respiratório,
requerem tratamento urgente por causa das altas taxas de morbidade e mortalidade.
OBJETIVOS
O objetivo deste protocolo é sistematizar a abordage

In [ ]:
import re

text = doc_ebserh.content
lines = text.splitlines()

def normalize_line(line: str) -> str:
    return re.sub(r'\s+', ' ', line.strip()).lower()

line_counts = {}
for line in lines:
    normalized = normalize_line(line)
    if normalized:
        line_counts[normalized] = line_counts.get(normalized, 0) + 1

repeated = {line: count for line, count in line_counts.items() if count >= 2}
for line, count in sorted(repeated.items(), key=lambda x: x[1], reverse=True):
    print(f"[{count}x] {line}")

[7x] manejo da crise do angioedema hereditário
[2x]  reconhecer o paciente com sinais de obstrução de via aérea alta e
[2x] acionar o médico.
